# SolarSense: inspect the synthetic benchmark

Run `python -m solarsense setup` from the repository root before opening this notebook. Select the same Python environment. All data is synthetic. The outputs below were executed during the implementation run.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd() if Path("reports/metrics.json").exists() else Path.cwd().parent
report = json.loads((root / "reports/metrics.json").read_text())
df = pd.read_csv(root / "data/generated/solar.csv")
print("Model:", report["model_version"])
print("Rows:", len(df))
print(df.groupby("split").agg(rows=("fault", "size"), sessions=("session", "nunique")))
assert df.groupby("session").split.nunique().max() == 1


Model: 127932dc8505
Rows: 12096
             rows  sessions
split                      
calibration  1728        12
test         2592        18
train        6048        42
validation   1728        12


## Check the contract
The classifier receives only transformed sensor/environment features, not station identity, scenario truth or future values.

In [2]:
from solarsense.features import features
X = features(df.head(6))
print(X.round(3).to_string(index=False))


 voltage_ratio  current_ratio  power_ratio  ac_efficiency  irradiance  surface_temp  temp_rise  load_ratio  humidity
         0.000          0.000        0.000          0.000       0.000         0.212      0.000       0.936     0.655
         0.913          0.965        0.881          0.983       0.014         0.216      0.009       0.936     0.644
         0.918          0.937        0.861          0.981       0.036         0.226      0.022       0.936     0.647
         0.922          0.976        0.901          0.981       0.065         0.232      0.039       0.936     0.632
         0.927          0.933        0.866          0.978       0.079         0.238      0.048       0.936     0.624
         0.927          0.924        0.857          0.984       0.110         0.248      0.067       0.936     0.629


## Compare models and probability methods
Validation chooses the probability method. A calibration experiment can lose; keep that result visible.

In [3]:
for name, result in report["baselines"].items():
    print(name, "validation F1:", round(result["validation"]["macro_f1"], 4), "test F1:", round(result["test"]["macro_f1"], 4))
print(json.dumps(report["calibration"], indent=2))
print("Coverage:", round(report["coverage"], 4))
print("Session bootstrap interval:", report["macro_f1_session_bootstrap_95"])


rules validation F1: 0.9294 test F1: 0.8339
random_forest validation F1: 0.9357 test F1: 0.9618
boosting validation F1: 0.9201 test F1: 0.9439
{
  "selected_on_validation": "raw_forest",
  "raw_validation_log_loss": 0.1106824098622923,
  "sigmoid_validation_log_loss": 0.44641410548497307,
  "raw_test_log_loss": 0.13634148862195755,
  "sigmoid_test_log_loss": 0.23036232406871932
}
Coverage: 0.9695
Session bootstrap interval: [0.8909411143736913, 1.0]


![Synthetic confusion matrix and reliability](../reports/evaluation.png)

## Inspect actual errors
These are held-out predictions, including errors. Filtering for accepted rows alone would hide abstentions.

In [4]:
predictions = pd.read_csv(root / "reports/test_predictions.csv")
errors = predictions[predictions.truth != predictions.prediction]
print("Errors:", len(errors), "of", len(predictions))
print(errors.head(10).to_string(index=False))


Errors: 61 of 2592
    session               observed_at           truth prediction  confidence  accepted
session-069 2026-03-11T04:05:00+00:00 partial_shading    soiling    0.901725      True
session-069 2026-03-11T04:10:00+00:00 partial_shading    soiling    0.906132      True
session-069 2026-03-11T04:15:00+00:00 partial_shading     normal    0.661297      True
session-069 2026-03-11T04:20:00+00:00 partial_shading    soiling    0.576578      True
session-069 2026-03-11T04:25:00+00:00 partial_shading     normal    0.784164      True
session-069 2026-03-11T04:30:00+00:00 partial_shading     normal    0.416182     False
session-069 2026-03-11T04:35:00+00:00 partial_shading     normal    0.442606     False
session-069 2026-03-11T04:40:00+00:00 partial_shading     normal    0.391182     False
session-069 2026-03-11T04:45:00+00:00 partial_shading     normal    0.382849     False
session-069 2026-03-11T04:55:00+00:00 partial_shading    soiling    0.468754     False


## Forecast against persistence
Forecast inputs contain only origin-time information. Fault injection schedules are not supplied to the model.

In [5]:
rows = []
for horizon, result in report["forecast"].items():
    metrics = result["test"]["all"]
    rows.append({"minutes": horizon, "forest_mae_w": metrics["forest_mae_w"], "persistence_mae_w": metrics["persistence_mae_w"], "selected": result["selected_on_validation"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))


minutes  forest_mae_w  persistence_mae_w selected
     15        11.911             13.281   forest
     30        19.225             24.549   forest
     60        30.138             43.233   forest


## Review the stress cases
Unknown detection is incomplete. An unfamiliar combination can still receive a known label. These counts must stay in the project discussion.

In [6]:
scenarios = json.loads((root / "reports/scenarios.json").read_text())
print(json.dumps(scenarios["stress_decisions"], indent=2))


{
  "normal": {
    "normal": 20
  },
  "partial_shading": {
    "partial_shading": 10,
    "normal": 7,
    "uncertain": 3
  },
  "soiling": {
    "uncertain": 1,
    "normal": 19
  },
  "inverter_fault": {
    "inverter_fault": 20
  },
  "unknown": {
    "uncertain": 1,
    "partial_shading": 19
  },
  "sensor_offset": {
    "normal": 20
  },
  "missing": {
    "data_quality": 20
  },
  "invalid": {
    "data_quality": 20
  },
  "night": {
    "inactive": 20
  },
  "cloud": {
    "normal": 20
  },
  "low_load": {
    "normal": 20
  }
}
